In [ ]:
# %%capture
# We're installing the latest Torch, Triton, OpenAI's Triton kernels, Transformers and Unsloth!
!pip install --upgrade -qqq uv
try: import numpy; install_numpy = f"numpy=={numpy.__version__}"
except: install_numpy = "numpy"
!uv pip install -qqq \
    "torch>=2.8.0" "triton>=3.5.0" {install_numpy} \
    "transformers>=4.56.0" \
    "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
    "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
    torchvision bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.2/21.2 MB 74.1 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Load model using Unsloth FastLanguageModel for function calling
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048  # Increased for function calling complexity
dtype = torch.float16  # Use float16 for Unsloth compatibility (not bfloat16)
model_name = "meta-llama/Llama-3.2-3B-Instruct"  # Changed to Llama 3 for better function calling

# Load model and tokenizer with Unsloth
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    dtype=dtype,
    max_seq_length=max_seq_length,
    load_in_4bit=True,  # Use 4-bit quantization to save memory
    full_finetuning=False,
)

print("✅ Model and tokenizer loaded successfully!")
print(f"Model: {model_name}")
print(f"Max sequence length: {max_seq_length}")
print(f"Using dtype: {dtype}")


In [ ]:
# ============================================================================
# UTILITY FUNCTIONS FOR FUNCTION CALLING FINE-TUNING WITH TOOL SUPPORT
# ============================================================================

import json
import re
import sys

def load_tools_from_definitions():
    """
    Load tool definitions from tool_definitions.py
    
    Returns:
        List of tool definitions or empty list if not found
    """
    try:
        # Try to import from the actual path
        sys.path.insert(0, '/content/drive/MyDrive/Colab Notebooks/AI/do an')
        from tool_definitions import INPUT_VALIDATION_TOOLS, INTENT_CLASSIFICATION_TOOLS, CLARIFICATION_TOOLS
        
        all_tools = []
        try:
            all_tools.extend(INPUT_VALIDATION_TOOLS if isinstance(INPUT_VALIDATION_TOOLS, list) else [INPUT_VALIDATION_TOOLS])
        except:
            pass
        try:
            all_tools.extend(INTENT_CLASSIFICATION_TOOLS if isinstance(INTENT_CLASSIFICATION_TOOLS, list) else [INTENT_CLASSIFICATION_TOOLS])
        except:
            pass
        try:
            all_tools.extend(CLARIFICATION_TOOLS if isinstance(CLARIFICATION_TOOLS, list) else [CLARIFICATION_TOOLS])
        except:
            pass
        
        print(f"✅ Loaded {len(all_tools)} tools from tool_definitions.py")
        return all_tools
    except Exception as e:
        print(f"⚠️ Could not load tools from tool_definitions.py: {e}")
        print("   Continuing without explicit tools...")
        return []


def extract_json_from_response(response: str) -> dict:
    """
    Extract JSON function call from model response
    
    Args:
        response: Model generated text
    
    Returns:
        Parsed JSON dict or None if no valid JSON found
    """
    # Try to find JSON block
    json_match = re.search(r'\{[^{}]*\}', response, re.DOTALL)
    if json_match:
        try:
            return json.loads(json_match.group())
        except json.JSONDecodeError:
            pass
    return None


def validate_function_call(response: str) -> bool:
    """
    Check if response contains valid function call JSON
    
    Returns:
        True if valid JSON with 'name' and 'parameters' fields
    """
    try:
        data = extract_json_from_response(response)
        if data and "name" in data and "parameters" in data:
            return True
    except:
        pass
    return False


def handle_function_calling(query: str, tokenizer, model, tools=None):
    """
    Handle function calling for market research tasks with tool support
    
    Args:
        query: User input with tool definitions
        tokenizer: Tokenizer for model
        model: Fine-tuned model for function calling
        tools: Tool definitions (optional)
    
    Returns:
        dict: Parsed function call or error
    """
    try:
        messages = [
            {
                "role": "system",
                "content": "You are a market research analyst. You must respond with valid JSON function calls."
            },
            {
                "role": "user",
                "content": query
            }
        ]
        
        # Apply chat template with tools support (matching actual usage)
        if hasattr(tokenizer, "apply_chat_template"):
            model_input = tokenizer.apply_chat_template(
                messages,
                tools=tools,  # Pass tools if available
                tokenize=False,
                add_generation_prompt=True,
            )
        else:
            # Fallback: use last user message as prompt
            user_msgs = [m.get("content", "") for m in messages if m.get("role") == "user"]
            model_input = user_msgs[-1] if user_msgs else query
        
        # Tokenize
        model_inputs = tokenizer([model_input], return_tensors="pt")
        
        if torch.cuda.is_available():
            model_inputs = model_inputs.to('cuda')
        
        # Generate response
        with torch.no_grad():
            generated_ids = model.generate(
                model_inputs.input_ids,
                max_new_tokens=500,
                temperature=0.3,  # Low temperature for JSON precision
                top_p=0.95,
                do_sample=True,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.eos_token_id,
            )
        
        # Decode response
        generated_ids = [
            output_ids[len(input_ids):]
            for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
        ]
        
        response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
        
        # Extract and validate JSON
        json_data = extract_json_from_response(response)
        is_valid = json_data is not None and "name" in json_data and "parameters" in json_data
        
        return {
            "success": is_valid,
            "response": response,
            "json_data": json_data,
            "valid_function_call": is_valid
        }
    
    except Exception as e:
        return {
            "success": False,
            "error": str(e),
            "valid_function_call": False
        }


def test_function_calling(tokenizer, model, tools=None, test_cases=None):
    """
    Test function calling with sample test cases
    
    Args:
        tokenizer: Tokenizer
        model: Fine-tuned model
        tools: Tool definitions (optional)
        test_cases: List of test prompts (optional)
    """
    if test_cases is None:
        test_cases = [
            'Xin chào! Bạn khỏe không?',
            'Phân tích thị trường cà phê specialty ở Việt Nam',
        ]
    
    print("\n" + "="*70)
    print("🧪 TESTING FUNCTION CALLING CAPABILITY")
    print("="*70 + "\n")
    
    success_count = 0
    for i, test_prompt in enumerate(test_cases, 1):
        print(f"Test Case {i}: {test_prompt[:60]}...")
        print("-" * 70)
        
        result = handle_function_calling(test_prompt, tokenizer, model, tools=tools)
        
        if result["valid_function_call"]:
            success_count += 1
            print("✅ VALID FUNCTION CALL")
            print(f"Function: {result['json_data'].get('name')}")
            print(f"Parameters: {json.dumps(result['json_data'].get('parameters'), indent=2, ensure_ascii=False)}")
        else:
            print("❌ INVALID FUNCTION CALL")
            print(f"Response: {result['response'][:200]}...")
        
        print()
    
    print("="*70)
    print(f"✅ SUCCESS RATE: {success_count}/{len(test_cases)} ({100*success_count//len(test_cases)}%)")
    print("="*70 + "\n")
    
    return success_count / len(test_cases)


print("✅ Function calling utilities loaded successfully!")

Utility functions loaded successfully!


In [ ]:
# ============================================================================
# SETUP LoRA FOR FUNCTION CALLING + LOAD TRAINING DATA WITH TOOL SUPPORT
# ============================================================================

# Setup LoRA optimized for function calling
model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # Higher rank for better function call accuracy
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,  # Higher alpha for stronger updates
    lora_dropout=0.0,  # Changed to 0 for Unsloth fast patching
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

print("✅ LoRA setup completed (optimized for function calling)!")

# Load ALL tool sets from tool_definitions (not just merged list)
print("\n🔧 Loading tool definitions...")
import sys
sys.path.insert(0, '/content/drive/MyDrive/Colab Notebooks/AI/do an')

try:
    from tool_definitions import (
        INPUT_VALIDATION_TOOLS,
        INTENT_CLASSIFICATION_TOOLS, 
        CLARIFICATION_TOOLS,
        PLANNING_TOOLS,
        REACT_TOOLS,
        GENERATE_SEARCH_QUERIES_TOOLS,
        SYNTHESIS_TOOLS,
        CAMPAIGN_TOOLS,
        SWOT_TOOLS
    )
    
    # Create mapping of tool_calls names to actual tool definitions
    tool_sets = {
        "INPUT_VALIDATION_TOOLS": INPUT_VALIDATION_TOOLS,
        "INTENT_CLASSIFICATION_TOOLS": INTENT_CLASSIFICATION_TOOLS,
        "CLARIFICATION_TOOLS": CLARIFICATION_TOOLS,
        "PLANNING_TOOLS": PLANNING_TOOLS,
        "REACT_TOOLS": REACT_TOOLS,
        "GENERATE_SEARCH_QUERIES_TOOLS": GENERATE_SEARCH_QUERIES_TOOLS,
        "SYNTHESIS_TOOLS": SYNTHESIS_TOOLS,
        "CAMPAIGN_TOOLS": CAMPAIGN_TOOLS,
        "SWOT_TOOLS": SWOT_TOOLS,
    }
    print(f"✅ Loaded {len(tool_sets)} tool sets from tool_definitions.py")
    for name in tool_sets.keys():
        print(f"   - {name}")
except Exception as e:
    print(f"⚠️ Could not load tool sets: {e}")
    tool_sets = {}

# Load training data
import json
from datasets import load_dataset

print("\n📥 Loading training data...")

# Load from training_data.json (the standardized format)
with open("/content/drive/MyDrive/Colab Notebooks/AI/do an/training_data.json", "r", encoding="utf-8") as f:
    raw_training_data = json.load(f)

print(f"✅ Loaded {len(raw_training_data)} training examples")

# Convert to messages format with tool set mapping
training_examples = []
for example_idx, example in enumerate(raw_training_data):
    # Get messages
    messages = None
    if "messages" in example:
        messages = example.get("messages", [])
    elif "conversation" in example:
        messages = example.get("conversation", [])
    
    if messages and len(messages) > 0:
        # Extract tool_calls value from tool role message
        tools_for_this_example = None
        
        # Filter out role="tool" messages and extract tool set name
        filtered_messages = []
        for msg in messages:
            if msg.get("role") == "tool":
                # Extract tool_calls name for mapping
                tool_calls_name = msg.get("tool_calls", "")
                if tool_calls_name in tool_sets:
                    tools_for_this_example = tool_sets[tool_calls_name]
                # Don't add tool role message to filtered list!
            else:
                # Keep all non-tool messages (system, user, assistant)
                filtered_messages.append(msg)
        
        training_examples.append({
            "messages": filtered_messages,  # Messages WITHOUT tool role
            "tools": tools_for_this_example  # Tool set stored separately
        })

print(f"✅ Converted {len(training_examples)} examples with tool mappings")

if len(training_examples) == 0:
    print("❌ WARNING: No training examples found!")
else:
    # Create dataset
    from datasets import Dataset
    dataset = Dataset.from_dict({
        "text": ["" for _ in training_examples]  # Will be filled by formatting
    })

    print(f"✅ Dataset created successfully")
    print(f"Example 1 structure: {len(training_examples[0]['messages'])} messages (tool messages removed)")
    for i, msg in enumerate(training_examples[0]['messages']):
        role = msg.get('role', 'unknown')
        content_preview = msg.get('content', '')[:50] if msg.get('content') else 'N/A'
        print(f"  - Message {i}: {role} - {content_preview}...")

    # Format using apply_chat_template with correct tools for each example
    def format_function_calling(idx):
        """
        Format example using Llama chat template.
        Tool messages have been removed, so we can safely pass tools parameter.
        """
        messages = training_examples[idx]['messages']
        example_tools = training_examples[idx]['tools']
        
        # Apply chat template with the specific tools for this example
        if hasattr(tokenizer, "apply_chat_template"):
            try:
                text = tokenizer.apply_chat_template(
                    messages,
                    tools=example_tools if example_tools else None,  # Pass specific tools
                    tokenize=False,
                    add_generation_prompt=False
                )
            except Exception as e:
                print(f"   ⚠️ Error formatting example {idx}: {e}")
                # Fallback: format without tools (messages already cleaned)
                try:
                    text = tokenizer.apply_chat_template(
                        messages,
                        tokenize=False,
                        add_generation_prompt=False
                    )
                except Exception as e2:
                    print(f"   ❌ Fallback also failed: {e2}")
                    # Last resort: manual formatting
                    text = ""
                    for msg in messages:
                        role = msg.get('role', 'unknown')
                        content = msg.get('content', '')
                        text += f"<{role}>{content}</{role}>\n"
        else:
            # Fallback: basic formatting
            text = ""
            for msg in messages:
                role = msg.get('role', 'unknown')
                content = msg.get('content', '')
                text += f"<{role}>{content}</{role}>\n"
        
        return text

    print("\n🔄 Formatting dataset with per-example tools (tool messages removed)...")
    texts = [format_function_calling(i) for i in range(len(training_examples))]
    dataset = Dataset.from_dict({"text": texts})

    print("✅ Dataset formatted successfully!")
    print(f"Sample formatted text length: {len(dataset[0]['text'])} characters")
    print(f"First 300 chars:\n{dataset[0]['text'][:300]}...")

    # Setup training configuration
    from trl import SFTConfig, SFTTrainer

    training_config = SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=2,
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=1e-4,
        max_steps=75,  # Stop training at maximum 75 steps
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir="./llama_function_calling_finetune",
        report_to="none",
        save_strategy="no",
        eval_strategy="no",  # No evaluation - dataset too small
        max_grad_norm=1.0,
    )

    print("\n✅ Training configuration ready!")
    print(f"   - Batch size: {training_config.per_device_train_batch_size}")
    print(f"   - Learning rate: {training_config.learning_rate}")
    print(f"   - Epochs: {training_config.num_train_epochs}")
    print(f"   - Dataset size: {len(dataset)}")
    print(f"   - Tool sets available: {len(tool_sets)}")

    # Create trainer
    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=dataset,
        args=training_config,
    )

    print("✅ SFTTrainer initialized successfully!")
    print("\n🚀 Ready to start training...")


Unsloth 2025.10.1 patched 22 layers with 22 QKV layers, 22 O layers and 22 MLP layers.


LoRA setup completed with Unsloth!


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/1888 [00:00<?, ? examples/s]

Dataset loaded and formatted successfully with 1888 samples
Sample formatted text:
<|system|>
You are Mia, a chatty AI with a friendly, laid-back vibe. Clarify vague questions briefly.</s>
<|user|>
What’s a cool place to go?</s>
<|assistant|>
Cool place for what? Cafe, beach, or hik...


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/200 [00:00<?, ? examples/s]

SFTTrainer setup completed successfully!


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 200 | Num Epochs = 1 | Total steps = 50
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 6,307,840 of 1,106,356,224 (0.57% trained)


Step,Training Loss
1,2.173900
2,2.036200
3,2.165200
4,1.925600
5,2.030100
6,1.915400
7,1.786800
8,1.619800
9,1.617300
10,1.599600


Training completed successfully!
Model saved successfully!


In [ ]:
# ============================================================================
# START TRAINING WITH TOOL SUPPORT
# ============================================================================

print("\n" + "="*70)
print("🚀 STARTING FINE-TUNING WITH TOOL-AWARE FUNCTION CALLING")
print("="*70 + "\n")

# Start training
trainer.train()

print("\n" + "="*70)
print("✅ TRAINING COMPLETED!")
print("="*70)

# Save the model
output_dir = "./llama_function_calling_finetune_final"
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"\n💾 Model saved to: {output_dir}")
print(f"   - Model weights saved")
print(f"   - Tokenizer saved")


In [ ]:
# ============================================================================
# PUSH FINE-TUNED MODEL TO HUGGING FACE HUB
# ============================================================================

from huggingface_hub import login, HfApi

print("\n" + "="*70)
print("🚀 PUSHING MODEL TO HUGGING FACE HUB")
print("="*70 + "\n")

# Login to Hugging Face (you need a write token)
# Get token from: https://huggingface.co/settings/tokens
print("📝 Logging into Hugging Face...")
print("   (You need a write access token from https://huggingface.co/settings/tokens)")

try:
    login()  # Will prompt for token
    print("✅ Logged in successfully!")
except Exception as e:
    print(f"⚠️ Login failed: {e}")
    print("   Skipping Hugging Face push...")
    exit()

# Push model to hub
repo_id = "justrandomname/function-calling"
print(f"\n📤 Pushing model to {repo_id}...")

try:
    # Push model weights
    model.push_to_hub(
        repo_id=repo_id,
        token=None,  # Uses login() token
        private=False,
        commit_message="Fine-tuned Llama-3.2-3B for function calling with tool support"
    )
    print(f"✅ Model pushed successfully!")
    
    # Push tokenizer
    print(f"📤 Pushing tokenizer...")
    tokenizer.push_to_hub(
        repo_id=repo_id,
        token=None,
        private=False,
    )
    print(f"✅ Tokenizer pushed successfully!")
    
    # Push training config
    print(f"📤 Pushing training config...")
    training_config.push_to_hub(
        repo_id=repo_id,
        token=None,
    )
    print(f"✅ Training config pushed successfully!")
    
    print("\n" + "="*70)
    print(f"🎉 SUCCESS! Model available at:")
    print(f"   https://huggingface.co/{repo_id}")
    print("="*70)
    
except Exception as e:
    print(f"❌ Push failed: {e}")
    print("   Make sure your token has write access to the repo")


In [ ]:
# ============================================================================
# TEST FINE-TUNED MODEL WITH TOOL SUPPORT
# ============================================================================

print("\n" + "="*70)
print("🧪 TESTING FINE-TUNED MODEL WITH TOOLS")
print("="*70 + "\n")

# Prepare model for inference
model = FastLanguageModel.for_inference(model)

# Test with tools
test_queries = [
    "Xin chào! Bạn khỏe không?",
    "Phân tích thị trường cà phê specialty ở Việt Nam",
    "Giáo dục online ở Việt Nam so với nước khác như thế nào?",
]

print(f"Testing with {len(tools)} tools loaded from tool_definitions\n")

for i, query in enumerate(test_queries, 1):
    print(f"\n{'='*70}")
    print(f"Test {i}: {query}")
    print(f"{'='*70}")
    
    result = handle_function_calling(query, tokenizer, model, tools=tools)
    
    if result["valid_function_call"]:
        print("✅ SUCCESS - Valid function call generated!")
        print(f"Function: {result['json_data'].get('name')}")
        params = result['json_data'].get('parameters', {})
        print(f"Parameters: {json.dumps(params, indent=2, ensure_ascii=False)}")
    else:
        print("⚠️ Response generated (may need more training)")
        print(f"Output: {result.get('response', '')[:300]}...")

print("\n" + "="*70)
print("✅ TESTING COMPLETE")
print("="*70)
